# Multi-Agent LLM Debate Collapse Detection — Demo Notebook

This notebook processes a real multi-agent LLM debate dataset to detect debate outcomes (converged, collapsed, or deadlocked). Built from the peer-reviewed Multi-Agent-LLMs/DEBATE corpus (EMNLP 2025 MALLM demo, HuggingFace), featuring Llama-3.3-70B agents with diverse personas debating factual questions.

**Dataset overview:**
- 95 debates (45 converged, 45 collapsed, 5 deadlocked)
- 665 round-level examples (3-7 rounds per debate)
- 3 debate configs: memory_simple_voting, debate_majority_consensus, relay_approval_voting
- Each example includes question, agent responses, agreement scores, and metadata

## Setup: Install Dependencies

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Only loguru is not pre-installed on Colab
_pip('loguru==0.7.2')

# Core packages: pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

## Imports

In [ ]:
import json
import random
import re
import sys
from collections import defaultdict
from pathlib import Path

from loguru import logger
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Setup logging for notebook
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

## Data Loading: GitHub + Local Fallback

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-eb7b29-testing-critical-slowing-down-as-an-earl/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub (Colab) or local file (dev)."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

## Load Demo Data

In [ ]:
data = load_data()
examples = data['datasets'][0]['examples']
logger.info(f"Loaded {len(examples)} round-level examples from demo dataset")

## Configuration: Tunable Parameters

All parameters are set to minimal demo values. Increase for full processing.

In [ ]:
# Demo configuration (minimal values)
RANDOM_SEED = 42
MAX_DEBATES_TO_ANALYZE = len(examples)  # Use all loaded examples
MIN_AGREEMENT_FOR_CONSENSUS = 0.66      # Fraction of agents sharing modal solution

logger.info(f"Config: SEED={RANDOM_SEED}, MAX_DEBATES={MAX_DEBATES_TO_ANALYZE}, MIN_AGREEMENT={MIN_AGREEMENT_FOR_CONSENSUS}")

## Solution Normalization

Normalize agent solution text for agreement scoring. Remove extra whitespace and truncate to first 50 chars for comparison.

In [ ]:
def normalize_solution(solution: str) -> str:
    """Normalize solution text: trim whitespace, lowercase, truncate to 50 chars."""
    return re.sub(r"\s+", " ", solution.strip().lower())[:50]

# Test normalization
test_solutions = [
    "A) Yes",
    "A)   Yes   with extra    spaces",
    "B) No"
]
logger.info("Normalization test:")
for sol in test_solutions:
    logger.info(f"  '{sol}' -> '{normalize_solution(sol)}'")

## Outcome Classification

Classify debate outcomes based on final-round agreement and decision correctness:
- **Converged**: correct consensus (high agreement + correct answer)
- **Collapsed**: wrong consensus (high agreement but wrong answer)
- **Deadlocked**: no consensus (low agreement in final round)

In [ ]:
def classify_outcome(final_round_agreement: float, decision_success: bool) -> str:
    """Classify debate outcome based on agreement and correctness."""
    if decision_success and final_round_agreement >= MIN_AGREEMENT_FOR_CONSENSUS:
        return "converged"
    if final_round_agreement >= MIN_AGREEMENT_FOR_CONSENSUS:
        return "collapsed"
    return "deadlocked"

# Demo classification
test_cases = [
    (0.95, True, "converged"),
    (0.80, False, "collapsed"),
    (0.50, True, "deadlocked")
]
logger.info("Classification test:")
for agree, success, expected in test_cases:
    result = classify_outcome(agree, success)
    status = "✓" if result == expected else "✗"
    logger.info(f"  {status} agree={agree}, success={success} -> {result} (expected {expected})")

## Process Examples: Extract Features and Metadata

For each example, parse JSON input and extract debate metadata, agreement scores, and labels.

In [ ]:
def process_examples(examples, max_count=None):
    """Process round-level examples and aggregate statistics."""
    processed = []
    errors = []
    
    for idx, ex in enumerate(examples[:max_count or len(examples)]):
        try:
            # Parse input JSON
            input_data = json.loads(ex['input'])
            
            # Extract fields
            example_processed = {
                'round_number': input_data['round_number'],
                'question_text': input_data['question_text'][:80],  # Truncate for display
                'num_agents': len(input_data.get('agent_responses', [])),
                'debate_id': ex.get('metadata_debate_id', 'unknown'),
                'outcome': ex['output'],
                'agreement_score': ex.get('metadata_agreement_score', 0.0),
                'decision_success': ex.get('metadata_decision_success', False),
                'source_config': ex.get('metadata_source_config', 'unknown'),
                'persona_diversity': ex.get('metadata_persona_diversity', 0.0)
            }
            processed.append(example_processed)
        except Exception as e:
            errors.append((idx, str(e)))
    
    if errors:
        logger.warning(f"Errors processing {len(errors)} examples: {errors[:3]}")
    
    return processed

# Process all examples
processed_examples = process_examples(examples, max_count=MAX_DEBATES_TO_ANALYZE)
logger.info(f"Processed {len(processed_examples)} examples")

## Statistics and Summary

Aggregate outcomes, agreement scores, and other metrics.

In [ ]:
# Aggregate statistics
outcomes = [ex['outcome'] for ex in processed_examples]
outcome_counts = pd.Series(outcomes).value_counts().to_dict()

agreement_scores = [ex['agreement_score'] for ex in processed_examples]
mean_agreement = np.mean(agreement_scores)
std_agreement = np.std(agreement_scores)

by_outcome = defaultdict(list)
for ex in processed_examples:
    by_outcome[ex['outcome']].append(ex)

logger.info(f"\nOutcome distribution:")
for outcome, exs in sorted(by_outcome.items()):
    logger.info(f"  {outcome}: {len(exs)} examples ({len(exs)/len(processed_examples)*100:.1f}%)")

logger.info(f"\nAgreement score statistics:")
logger.info(f"  Mean: {mean_agreement:.3f}")
logger.info(f"  Std Dev: {std_agreement:.3f}")
logger.info(f"  Min: {np.min(agreement_scores):.3f}")
logger.info(f"  Max: {np.max(agreement_scores):.3f}")

## Group by Config and Outcome

Analyze patterns across debate configs and outcomes.

In [ ]:
# Group by config
by_config = defaultdict(list)
for ex in processed_examples:
    by_config[ex['source_config']].append(ex)

logger.info(f"\nExamples per config:")
for config, exs in sorted(by_config.items()):
    logger.info(f"  {config}: {len(exs)} examples")

# Outcome distribution per config
logger.info(f"\nOutcome distribution per config:")
for config in sorted(by_config.keys()):
    outcomes_for_config = [ex['outcome'] for ex in by_config[config]]
    outcome_dist = pd.Series(outcomes_for_config).value_counts().to_dict()
    logger.info(f"  {config}:")
    for outcome, count in sorted(outcome_dist.items()):
        logger.info(f"    {outcome}: {count}")

## Visualization: Results Summary

Plot outcome distribution and agreement scores by outcome type.

In [ ]:
# Create summary plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Outcome distribution
outcome_labels = list(outcome_counts.keys())
outcome_values = list(outcome_counts.values())
colors = {'converged': '#2ecc71', 'collapsed': '#e74c3c', 'deadlocked': '#95a5a6'}
bar_colors = [colors.get(label, '#3498db') for label in outcome_labels]

axes[0].bar(outcome_labels, outcome_values, color=bar_colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Debate Outcome Distribution', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(outcome_values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Plot 2: Agreement scores by outcome
for outcome in sorted(by_outcome.keys()):
    scores = [ex['agreement_score'] for ex in by_outcome[outcome]]
    axes[1].scatter([outcome]*len(scores), scores, alpha=0.6, s=80, label=outcome, color=colors.get(outcome, '#3498db'))

axes[1].axhline(y=MIN_AGREEMENT_FOR_CONSENSUS, color='red', linestyle='--', linewidth=2, label=f'Consensus threshold ({MIN_AGREEMENT_FOR_CONSENSUS})')
axes[1].set_ylabel('Agreement Score', fontsize=11)
axes[1].set_title('Agreement Scores by Outcome', fontsize=12, fontweight='bold')
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(axis='y', alpha=0.3)
axes[1].legend(loc='best', fontsize=9)

plt.tight_layout()
plt.show()

logger.info("Visualization complete.")

## Sample Examples

Display examples from each outcome category for manual inspection.

In [ ]:
# Show sample examples from each outcome
logger.info("\nSample examples from each outcome:")
for outcome in sorted(by_outcome.keys()):
    exs = by_outcome[outcome]
    if exs:
        sample = exs[0]
        logger.info(f"\n{outcome.upper()}:")
        logger.info(f"  Question: {sample['question_text']}")
        logger.info(f"  Debate ID: {sample['debate_id']}")
        logger.info(f"  Round: {sample['round_number']}")
        logger.info(f"  # Agents: {sample['num_agents']}")
        logger.info(f"  Agreement: {sample['agreement_score']:.3f}")
        logger.info(f"  Decision Success: {sample['decision_success']}")
        logger.info(f"  Config: {sample['source_config']}")
        logger.info(f"  Persona Diversity: {sample['persona_diversity']:.3f}")

## Summary

This demo notebook successfully:
1. Loaded the multi-agent LLM debate dataset from GitHub (or local fallback)
2. Processed round-level examples and extracted features
3. Classified debate outcomes using agreement scores and decision correctness
4. Analyzed patterns across debate configs and outcome types
5. Visualized the outcome distribution and agreement patterns

**To scale to full dataset:**
- Increase `MAX_DEBATES_TO_ANALYZE` in the config cell
- Load the full dataset from `full_data_out.json` instead of mini demo data
- Runtime scales linearly with number of examples (currently demo with minimal data)

In [ ]:
logger.info(f"\n✓ Demo notebook complete!")
logger.info(f"  Total examples processed: {len(processed_examples)}")
logger.info(f"  Outcome labels: {', '.join(sorted(set(outcomes)))}")
logger.info(f"  Mean agreement score: {mean_agreement:.3f}")
logger.info(f"  Dataset size on disk: {Path('mini_demo_data.json').stat().st_size / 1024:.1f} KB")